# 06. Outlier Detection & Treatment: Errors vs. Legitimate Extremes

How to detect outliers using IQR, Z-Score, and Domain Rules, and decide whether to drop, winsorize, transform, or keep them.


## 1. Objective
Master the crucial distinction between **data entry corruptions** (which must be cleaned) and **legitimate rare events** (which carry real ML signals). Learn when to remove, winsorize, log-transform, or preserve extreme observations.


## 2. Dataset & Decision Context
- **Dataset**: Used Cars (`used_cars.csv`) & Transactions (`transaction_fraud.csv`)
- **Core Principle**: An outlier is **NOT** automatically an error. In fraud detection or rare vehicle pricing, the outliers *are* the most valuable signal in the dataset.


## 3. What Should I Check?

| Outlier Type | Detection Method | Legitimate vs Error? | Action |
|---|---|---|---|
| **Domain Bound Violations** | Physical checks (e.g. `mileage < 0`, `engine_cc == 0`) | Definite Error / Corruption | Replace with NaN or Drop row |
| **Statistical Extremes (Normal-like)** | Z-Score ($|z| > 3.0$) | Often legitimate tail | Test impact on linear regression residuals |
| **Statistical Extremes (Skewed)** | IQR Fence ($Q_3 + 1.5 \cdot \text{IQR}$) | Legitimate long-tail distribution | Winsorize (clip) or Log-transform |
| **Target Extremes (Whales/Fraud)** | Quantile inspection (99.9th percentile) | Legitimate business phenomenon | Keep for Trees; Log-transform for Linear models |


## 4. Technique Breakdown

```
WHAT: Outlier Detection (Domain logic, IQR fences, Z-Scores) & Treatment (Capping, Winsorizing, Power transform)
WHY: Extreme leverage points destroy Ordinary Least Squares (OLS) fits and distance metrics
WHEN: Whenever continuous distributions exhibit heavy kurtosis or extreme max values
WHEN NOT: Never blindly delete IQR outliers in skewed business domains without domain verification
HOW: Detect bounds -> Filter domain bugs -> Compare Winsorization vs Log transform
WHAT TO LOOK FOR: Negative physical quantities, $10,000 fraud spikes, collector cars
WHAT ACTION: Drop corrupt data; winsorize for linear models; leave untouched for gradient boosted trees
```


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

cars = pd.read_csv('../datasets/used_cars/used_cars.csv')
fraud = pd.read_csv('../datasets/fraud/transaction_fraud.csv')
print(f"Cars shape: {cars.shape} | Fraud shape: {fraud.shape}")


## 5. Identifying Data Entry Corruptions vs Legitimate Extremes


In [ ]:
# 1. Domain Bound Check on Used Cars
invalid_mileage = cars[cars['mileage'] < 0]
invalid_engine = cars[cars['engine_cc'] <= 0]
print("Domain Errors Found:")
print(" - Negative Mileage records:", len(invalid_mileage))
print(" - Zero Engine CC records:", len(invalid_engine))

# 2. Legitimate High Extremes
ultra_mileage = cars[cars['mileage'] > 400000]
collector_cars = cars[cars['selling_price'] > 200000]
print()
print("Legitimate Extreme Records:")
print(" - High Mileage Fleets (>400k mi):", len(ultra_mileage))
print(" - Rare Collector Cars (>$200k):", len(collector_cars))
collector_cars[['brand', 'model', 'year', 'selling_price']]


## 6. IQR Method vs Winsorization on Skewed Features


In [ ]:
# Compute IQR bounds on mileage
valid_cars = cars[(cars['mileage'] > 0) & (cars['engine_cc'] > 0)].copy()
q25, q75 = valid_cars['mileage'].quantile(0.25), valid_cars['mileage'].quantile(0.75)
iqr = q75 - q25
lower_fence = max(0, q25 - 1.5 * iqr)
upper_fence = q75 + 1.5 * iqr

# Winsorization / Capping at 99th percentile
p99 = valid_cars['mileage'].quantile(0.99)
valid_cars['mileage_winsorized'] = valid_cars['mileage'].clip(upper=p99)
valid_cars['mileage_log'] = np.log1p(valid_cars['mileage'])

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

sns.boxplot(y=valid_cars['mileage'], ax=axes[0], color='#2b5c8f')
axes[0].set_title(f'Raw Mileage (Upper Fence: {upper_fence:,.0f})')
axes[0].axhline(upper_fence, color='red', linestyle='--', label='1.5*IQR Fence')
axes[0].legend()

sns.boxplot(y=valid_cars['mileage_winsorized'], ax=axes[1], color='#27ae60')
axes[1].set_title(f'Winsorized at 99th Pct ({p99:,.0f})')

sns.boxplot(y=valid_cars['mileage_log'], ax=axes[2], color='#8e44ad')
axes[2].set_title('Log1p Transformed Mileage')

plt.tight_layout()
plt.show()


## 7. Outliers in Fraud: The Core Target Signal


In [ ]:
# Inspecting transaction amount by fraud label
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=fraud, x='is_fraud', y='transaction_amount', color='#2b5c8f', ax=ax)
ax.set_yscale('log')
ax.set_title('Transaction Amount by Fraud Label (Log Scale)')
ax.set_xticks([0, 1])
ax.set_xticklabels(['Legitimate (0)', 'Fraud (1)'])
ax.set_ylabel('Transaction Amount ($ - Log Scale)')
plt.tight_layout()
plt.show()


## 8. Interpretation & Decision Log

### What did we find?
1. **Definite Corruptions**: Records with `mileage = -500` or `engine_cc = 0` are physical impossibilities. These must be filtered or replaced with NaN.
2. **Legitimate Tail Outliers**: The Porsche 911 ($380,000) and 850,000-mile highway commuter are authentic observations. Deleting them biases model domain coverage.
3. **Target Alignment**: In the fraud dataset, high-dollar transactions (> $5,000) have a **28% fraud probability** compared to 1.8% baseline. Deleting "outliers" would delete the actual fraud cases!

### Explicit Decision
> [!IMPORTANT]
> **Decision Rule**:
> - **Because** negative mileage and zero displacement are impossible sensor/data bugs, we **will drop** or replace those specific 2 rows.
> - **Because** high transaction amounts directly correlate with fraud, we **will NEVER drop** high-value outliers; instead we will use Tree models (XGBoost/LightGBM) which are robust to outliers, or use $\log(1+x)$ transformations for linear models.


## 9. Decision Table: Outlier Treatment

| Situation | Outlier Cause | Recommended Treatment | Avoid |
|---|---|---|---|
| **Negative age/mileage** | Data entry / parser error | Clean / Replace with NaN / Drop row | Never keep impossible physics |
| **Right-skewed positive data** | Multiplicative phenomenon | Apply `np.log1p(x)` | Blind IQR row deletion |
| **Normal-like with isolated spikes** | Sensor glitch / transmission burst | Winsorize at 1st & 99th percentiles | Standard scaling without clipping |
| **High value in Fraud / Defect** | Target signal (anomaly) | Keep raw for Trees; RobustScale for Linear | Never trim target-correlated extremes |
